In [1]:
#using baynesian search for hyperparameter tuning

In [2]:
#learning optuna for bynesian

In [3]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.7 MB/s eta 0:00:00


In [4]:
#using indian diabetes dataset

In [5]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import pandas as pd

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names = columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [8]:
df.isnull().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [9]:
import numpy as np

#Replace zero values with NaN in columns where zero is not a valid value
cols_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_missing_vals] = df[cols_missing_vals].replace(0, np.nan)

#Impute missing values with mean of respective column
df.fillna(df.mean(), inplace = True)

#Checking for remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [10]:
X = df.drop('Outcome', axis = 1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)

scaler = StandardScaler()
X_train  = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'Training set shape:{X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape:(537, 8)
Test set shape: (231, 8)


In [30]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

#Defining objective function
def objective(trial):
  #Suggestive hyperparameters with values
  n_estimators = trial.suggest_int('n_estimators', 50, 200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  #Create the RandomForestClassifier with suggested hyperparameters
  model = RandomForestClassifier(
      n_estimators = n_estimators,
      max_depth = max_depth,
      random_state = 42
  )

  #Perform 3 fold cross validation and calculate accuracy
  score = cross_val_score(model, X_train, y_train, cv = 3, scoring = 'accuracy').mean()
  #returning the accuracy score for optuna to optimize
  return score

In [31]:
#Create a study object and optimize the objective function
study = optuna.create_study(direction = 'maximize', sampler = optuna.samplers.TPESampler())
#creating 50 trials to find best hyperparameters
study.optimize(objective, n_trials = 50)

[I 2026-04-24 13:08:25,241] A new study created in memory with name: no-name-612d2b58-4b32-4bb6-a01b-6c5221ef64d5
[I 2026-04-24 13:08:26,058] Trial 0 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 131, 'max_depth': 13}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-04-24 13:08:26,648] Trial 1 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 96, 'max_depth': 17}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-04-24 13:08:27,483] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 131, 'max_depth': 13}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-04-24 13:08:28,553] Trial 3 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 180, 'max_depth': 8}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-04-24 13:08:29,005] Trial 4 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 72, 'max_depth': 6}. Best is trial 0 with value: 0.77281191

In [32]:
print(f"Best trial accuracy: {study.best_trial.value}")
print(f"Best Hyperparameters: {study.best_trial.params}")

Best trial accuracy: 0.7821229050279329
Best Hyperparameters: {'n_estimators': 119, 'max_depth': 19}


In [33]:
from sklearn.metrics import accuracy_score

#training RandomForestClassifier using the best hyperparameter from optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state = 42)

#Fit the model to the training data
best_model.fit(X_train, y_train)

#Make predictions on the test set
y_pred = best_model.predict(X_test)

#Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

In [34]:
print(f"Test Accuracy with best hyperparameters: {test_accuracy:.2f}")

Test Accuracy with best hyperparameters: 0.74


In [17]:
#using randomizedSearch for hyperparameters

In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

#Defining objective function
def objective(trial):
  #Suggestive hyperparameters with values
  n_estimators = trial.suggest_int('n_estimators', 50, 200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  #Create the RandomForestClassifier with suggested hyperparameters
  model = RandomForestClassifier(
      n_estimators = n_estimators,
      max_depth = max_depth,
      random_state = 42
  )

  #Perform 3 fold cross validation and calculate accuracy
  score = cross_val_score(model, X_train, y_train, cv = 3, scoring = 'accuracy').mean()
  #returning the accuracy score for optuna to optimize
  return score

In [19]:
#Create a study object and optimize the objective function
study = optuna.create_study(direction = 'maximize', sampler = optuna.samplers.RandomSampler())
#creating 50 trials to find best hyperparameters
study.optimize(objective, n_trials = 50)

[I 2026-04-24 12:57:18,572] A new study created in memory with name: no-name-9715ec26-148b-4956-ae49-1677b55952dd
[I 2026-04-24 12:57:19,028] Trial 0 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 65, 'max_depth': 13}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-04-24 12:57:19,525] Trial 1 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 69, 'max_depth': 8}. Best is trial 1 with value: 0.7709497206703911.
[I 2026-04-24 12:57:20,561] Trial 2 finished with value: 0.7765363128491619 and parameters: {'n_estimators': 169, 'max_depth': 7}. Best is trial 2 with value: 0.7765363128491619.
[I 2026-04-24 12:57:21,162] Trial 3 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 94, 'max_depth': 11}. Best is trial 2 with value: 0.7765363128491619.
[I 2026-04-24 12:57:21,829] Trial 4 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 71, 'max_depth': 6}. Best is trial 2 with value: 0.77653631284

In [20]:
print(f"Best trial accuracy: {study.best_trial.value}")
print(f"Best Hyperparameters: {study.best_trial.params}")

Best trial accuracy: 0.7765363128491621
Best Hyperparameters: {'n_estimators': 114, 'max_depth': 20}


In [21]:
from sklearn.metrics import accuracy_score

#training RandomForestClassifier using the best hyperparameter from optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state = 42)

#Fit the model to the training data
best_model.fit(X_train, y_train)

#Make predictions on the test set
y_pred = best_model.predict(X_test)

#Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

print(f"Test Accuracy with best hyperparameters: {test_accuracy:.2f}")

Test Accuracy with best hyperparameters: 0.75


In [22]:
#using gridsearchCV for hyperparameter tuning

In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

#Defining objective function
def objective(trial):
  #Suggestive hyperparameters with values
  n_estimators = trial.suggest_int('n_estimators', 50, 200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  #Create the RandomForestClassifier with suggested hyperparameters
  model = RandomForestClassifier(
      n_estimators = n_estimators,
      max_depth = max_depth,
      random_state = 42
  )

  #Perform 3 fold cross validation and calculate accuracy
  score = cross_val_score(model, X_train, y_train, cv = 3, scoring = 'accuracy').mean()
  #returning the accuracy score for optuna to optimize
  return score

In [24]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth' : [5, 10, 15, 20]
}

In [25]:
#Create a study object and optimize the objective function
study = optuna.create_study(direction = 'maximize', sampler = optuna.samplers.RandomSampler())
#creating 50 trials to find best hyperparameters
study.optimize(objective, n_trials = 50)

[I 2026-04-24 13:00:35,788] A new study created in memory with name: no-name-8340bd61-7d79-401a-b487-c8b8078dccdc
[I 2026-04-24 13:00:36,644] Trial 0 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 117, 'max_depth': 6}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-04-24 13:00:37,393] Trial 1 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 111, 'max_depth': 7}. Best is trial 1 with value: 0.7746741154562384.
[I 2026-04-24 13:00:38,403] Trial 2 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 150, 'max_depth': 16}. Best is trial 1 with value: 0.7746741154562384.
[I 2026-04-24 13:00:39,497] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 142, 'max_depth': 11}. Best is trial 1 with value: 0.7746741154562384.
[I 2026-04-24 13:00:40,437] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 135, 'max_depth': 11}. Best is trial 1 with value: 0.774674

In [ ]:
print(f"Best trial accuracy: {study.best_trial.value}")
print(f"Best Hyperparameters: {study.best_trial.params}")

In [26]:
from sklearn.metrics import accuracy_score

#training RandomForestClassifier using the best hyperparameter from optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state = 42)

#Fit the model to the training data
best_model.fit(X_train, y_train)

#Make predictions on the test set
y_pred = best_model.predict(X_test)

#Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

print(f"Test Accuracy with best hyperparameters: {test_accuracy:.2f}")

Test Accuracy with best hyperparameters: 0.75


Visulazation in optuna

In [27]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [37]:
plot_optimization_history(study).show()

In [38]:
plot_parallel_coordinate(study).show()

In [35]:
plot_slice(study).show()

In [36]:
plot_contour(study).show()

In [39]:
plot_param_importances(study).show()

In [40]:
#optimizing multiple ml models

In [41]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [44]:
def objective(trial):
  classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

  if classifier_name == 'SVM':
    c = trial.suggest_float('C', 0.1, 100, log = True)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])
    model = SVC(C=c, kernel=kernel, gamma = gamma,  random_state=42)

  elif classifier_name == 'RandomForest':
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])

    model = RandomForestClassifier(
        n_estimators= n_estimators,
        max_depth = max_depth,
        min_samples = min_samples,
        min_samples_leaf = min_samples_leaf,
        bootstrap = bootstrap,
        random_state = 42
    )

  elif classifier_name == 'GradientBoosting':
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log = True)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_split', 1, 10)

    model = GradientBoostingClassifier(
        n_estimators = n_estimators,
        learning_rate = learning_rate,
        min_samples_split = min_samples_split,
        max_depth = max_depth,
        min_samples_leaf = min_samples_leaf,
        random_state = 42
    )


    score = cross_val_score(model, X_train, y_train, cv = 3, scoring = 'accuracy').mean()
    return score


In [45]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials = 100)

[I 2026-04-24 14:25:50,329] A new study created in memory with name: no-name-daa600b2-2067-4486-98a5-e4257a9e0da1
/tmp/ipykernel_3969/427246939.py:31: RuntimeWarning:

Inconsistent parameter values for distribution with name "min_samples_split"! This might be a configuration mistake. Optuna allows to call the same distribution with the same name more than once in a trial. When the parameter values are inconsistent optuna only uses the values of the first call and ignores all following. Using these values: {'log': False, 'step': 1, 'low': 2, 'high': 10}

[I 2026-04-24 14:25:54,137] Trial 0 finished with value: 0.7318435754189944 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 239, 'learning_rate': 0.03733545507033357, 'max_depth': 18, 'min_samples_split': 7}. Best is trial 0 with value: 0.7318435754189944.
[W 2026-04-24 14:25:54,140] Trial 1 failed with parameters: {'classifier': 'RandomForest', 'n_estimators': 225, 'max_depth': 6, 'min_samples_split': 2, 'min_samples

TypeError: RandomForestClassifier.__init__() got an unexpected keyword argument 'min_samples'